In [0]:

RAW_DIR = "dbfs:/Volumes/workspace/default/mlops_project/raw"
dbutils.fs.ls(RAW_DIR)

[FileInfo(path='dbfs:/Volumes/workspace/default/mlops_project/raw/loan.csv', name='loan.csv', size=1189395649, modificationTime=1767609417000)]

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("CREATE SCHEMA IF NOT EXISTS mlops_project")
# 1) Point to the raw CSV in your Unity Catalog Volume
csv_path = "dbfs:/Volumes/workspace/default/mlops_project/raw/*.csv"  

# 2) Read raw (no heavy transforms in Bronze)
bronze_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .option("multiLine", True)   # safe for some messy CSVs
         .option("escape", "\"")
         .csv(csv_path)
         .withColumn("_ingest_ts", F.current_timestamp())
         .withColumn("_source_file", F.col("_metadata.file_path"))
)


In [0]:
BRONZE_TBL = "mlops_project.lendingclub_bronze"

# create/append
(bronze_df.write.format("delta")
      .mode("overwrite") # could be append on the use case
      .saveAsTable(BRONZE_TBL))

